In [11]:
from __future__ import annotations

import json
import os
import time
import argparse
import csv
from dataclasses import dataclass, field, asdict
from typing import Any

from dotenv import load_dotenv
import anthropic
import openai
from concurrent.futures import ThreadPoolExecutor, as_completed

from prompts import (
    FRAMEWORKS,
    decomposer_prompts,
    mapper_prompts,
    reconciler_prompts,
)

load_dotenv()

True

API CLIENTS


In [2]:
_claude_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
_gpt_client    = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

CLAUDE_MODEL = "claude-sonnet-4-5"
GPT_MODEL    = "gpt-4o"


Data Types

In [3]:
@dataclass
class EvidenceUnit:
    id: str
    evidence_type: str
    relevance_tier: str
    evidence_text: str

@dataclass
class FrameworkMapping:
    framework_key: str
    domains: list[str]
    confidence: float
    rationale: str
    flags: list[str]
    reconciliation_note: str | None = None

@dataclass
class MappingResult:
    evidence_id: str
    evidence_text: str
    decomposed: dict
    mappings: dict[str, FrameworkMapping]
    model_agreement: dict[str, bool]
    audit: dict = field(default_factory=dict)   # raw model outputs

In [ ]:
API HELPERS

In [4]:
def _call_claude(system: str, user: str, max_tokens: int = 1200, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            msg = _claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=max_tokens,
                system=system,
                messages=[{"role": "user", "content": user}],
            )
            return msg.content[0].text
        except anthropic.RateLimitError:
            wait = 2 ** attempt
            print(f"    [rate limit] Claude — waiting {wait}s")
            time.sleep(wait)
    raise RuntimeError("Claude rate limit exceeded after retries")

def _call_gpt(system: str, user: str, max_tokens: int = 1200, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            resp = _gpt_client.chat.completions.create(
                model=GPT_MODEL,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": user},
                ],
            )
            return resp.choices[0].message.content
        except openai.RateLimitError:
            wait = 2 ** attempt
            print(f"    [rate limit] GPT — waiting {wait}s")
            time.sleep(wait)
    raise RuntimeError("GPT rate limit exceeded after retries")

def _safe_json(raw: str, label: str) -> dict:
    """Parse JSON from model output; strip markdown fences if present."""
    text = raw.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError as e:
        print(f"    [JSON parse error] {label}: {e}\n    Raw: {raw[:200]}")
        return {"domains": [], "confidence": 0.0, "rationale": "parse_error", "flags": ["parse_error"]}

def _domains_agree(a: dict, b: dict) -> bool:
    """True if the two mapper outputs share at least one primary domain."""
    return bool(set(a.get("domains", [])) & set(b.get("domains", [])))



PROMPT 3Ai - DECOMPOSITION

In [5]:

def decompose(unit: EvidenceUnit) -> dict:
    sys_p, usr_p = decomposer_prompts(
        unit.evidence_type, unit.relevance_tier, unit.evidence_text
    )
    raw = _call_claude(sys_p, usr_p, max_tokens=800)
    return _safe_json(raw, f"decompose/{unit.id}")

PROMPT 3Aii - FRAMEWORK MAPPING 

In [6]:
def map_framework(
    unit: EvidenceUnit,
    decomposed: dict,
    framework_key: str,
) -> tuple[dict, dict]:
    """Returns (claude_output, gpt_output) for one framework."""
    sys_p, usr_p = mapper_prompts(framework_key, unit.evidence_text, decomposed)
    raw_claude = _call_claude(sys_p, usr_p, max_tokens=600)
    raw_gpt    = _call_gpt(sys_p, usr_p, max_tokens=600)
    return (
        _safe_json(raw_claude, f"mapper_claude/{framework_key}/{unit.id}"),
        _safe_json(raw_gpt,    f"mapper_gpt/{framework_key}/{unit.id}"),
    )

def map_snomed(
    unit: EvidenceUnit,
    decomposed: dict,
) -> tuple[dict, dict]:
    """
    Single-pass SNOMED mapping against the flat 13-domain list.
    Returns (claude_output, gpt_output).
    """
    sys_p, usr_p = mapper_prompts("snomed", unit.evidence_text, decomposed)
    raw_claude = _call_claude(sys_p, usr_p, max_tokens=600)
    raw_gpt    = _call_gpt(sys_p, usr_p, max_tokens=600)
    return (
        _safe_json(raw_claude, f"snomed_claude/{unit.id}"),
        _safe_json(raw_gpt,    f"snomed_gpt/{unit.id}"),
    )

PROMPT 3Aiii- RECONCILIATION

In [7]:
def reconcile(
    unit: EvidenceUnit,
    framework_key: str,
    claude_out: dict,
    gpt_out: dict,
) -> dict:
    fw_name = FRAMEWORKS[framework_key]["name"]
    sys_p, usr_p = reconciler_prompts(
        framework_name=fw_name,
        evidence_text=unit.evidence_text,
        model_a_name="Claude",
        model_a_output=claude_out,
        model_b_name="GPT",
        model_b_output=gpt_out,
    )
    raw = _call_claude(sys_p, usr_p, max_tokens=700)
    return _safe_json(raw, f"reconcile/{framework_key}/{unit.id}")

RESULT ASSEMBLY

In [8]:
def _assemble_mapping(
    framework_key: str,
    reconciled: dict,
    agreed: bool,
) -> FrameworkMapping:
    return FrameworkMapping(
        framework_key=framework_key,
        domains=reconciled.get("domains", []),
        confidence=reconciled.get("confidence", 0.0),
        rationale=reconciled.get("rationale", ""),
        flags=reconciled.get("flags", []),
        reconciliation_note=reconciled.get("reconciliation_note"),
    )

EVIDENCE UNIT PIPELINE

In [9]:
def print_mapping_live(result: MappingResult) -> None:
    """Prints the mapping for a single EU immediately after it is processed."""
    print(f"\n{'─'*60}")
    print(f"  {result.evidence_id}: {result.evidence_text[:80]}")
    print(f"{'─'*60}")
    for fw_key, m in result.mappings.items():
        agreed  = "✓ agree" if result.model_agreement.get(fw_key) else "△ reconciled"
        domains = ", ".join(m.domains) if m.domains else "— none —"
        print(f"  [{fw_key:<20}]  conf={m.confidence:.2f}  {agreed}")
        print(f"    domains  : {domains}")
        print(f"    rationale: {m.rationale[:120]}")
        if m.flags and m.flags != ["not_applicable"]:
            print(f"    flags    : {m.flags}")
        if m.reconciliation_note:
            print(f"    recon    : {m.reconciliation_note[:120]}")

In [12]:
def process_unit(unit: EvidenceUnit) -> MappingResult:
    print(f"\n[{unit.id}] '{unit.evidence_text[:60]}...'")

    # ── Stage 1 (must complete first — feeds all mappers)
    print(f"  Stage 1: Decomposing...")
    decomposed = decompose(unit)

    mappings:  dict[str, FrameworkMapping] = {}
    agreement: dict[str, bool]             = {}
    audit:     dict                        = {"decomposed": decomposed, "raw": {}}

    all_fw = ["f13fv", "broad_domains", "bps_4p", "stipo", "snomed"]

    # ── Stage 2: fire all 10 mapper calls in parallel
    print(f"  Stage 2: Mapping all frameworks in parallel...")

    def run_mapper(fw_key):
        c_out, g_out = map_framework(unit, decomposed, fw_key)
        return fw_key, c_out, g_out

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(run_mapper, fw): fw for fw in all_fw}
        raw_outputs = {}
        for future in as_completed(futures):
            fw_key, c_out, g_out = future.result()
            raw_outputs[fw_key] = (c_out, g_out)
            print(f"    ✓ {fw_key}")

    # ── Stage 3: reconcile disagreements (sequential, cheap — only fires on disagreement)
    for fw_key in all_fw:
        c_out, g_out = raw_outputs[fw_key]
        agreed = _domains_agree(c_out, g_out)
        agreement[fw_key] = agreed
        audit["raw"][fw_key] = {"claude": c_out, "gpt": g_out}

        if agreed:
            reconciled = {**c_out, "confidence": round((c_out.get("confidence", 0) + g_out.get("confidence", 0)) / 2, 2)}
        else:
            print(f"    Disagreement on {fw_key} — reconciling...")
            reconciled = reconcile(unit, fw_key, c_out, g_out)

        mappings[fw_key] = _assemble_mapping(fw_key, reconciled, agreed)

    result = MappingResult(
        evidence_id=unit.id,
        evidence_text=unit.evidence_text,
        decomposed=decomposed,
        mappings=mappings,
        model_agreement=agreement,
        audit=audit,
    )
    print_mapping_live(result)
    return result

Batch Processing

In [13]:
def process_batch(units: list[EvidenceUnit]) -> list[MappingResult]:
    results = []
    for i, unit in enumerate(units, 1):
        print(f"\n{'='*60}\n[{i}/{len(units)}]")
        try:
            results.append(process_unit(unit))
        except Exception as e:
            print(f"  ERROR on {unit.id}: {e}")
    return results

Output helpers

In [18]:
def result_to_dict(r: MappingResult) -> dict:
    return {
        "evidence_id":     r.evidence_id,
        "evidence_text":   r.evidence_text,
        "decomposed":      r.decomposed,
        "model_agreement": r.model_agreement,
        "mappings": {
            k: {
                "domains":             m.domains,
                "confidence":          m.confidence,
                "rationale":           m.rationale,
                "flags":               m.flags,
                "reconciliation_note": m.reconciliation_note,
            }
            for k, m in r.mappings.items()
        },
    }

def save_json(results: list[MappingResult], path: str = "mappings_output.json") -> None:
    with open(path, "w") as f:
        json.dump([result_to_dict(r) for r in results], f, indent=2)
    print(f"\nSaved {len(results)} results → {path}")

def save_csv(results: list[MappingResult], path: str = "mappings_output.csv") -> None:
    """Flat CSV: one row per (evidence_unit × framework)."""
    rows = []
    for r in results:
        for fw_key, m in r.mappings.items():
            rows.append({
                "evidence_id":          r.evidence_id,
                "evidence_text":        r.evidence_text[:120],
                "framework":            fw_key,
                "domains":              "|".join(m.domains),
                "confidence":           m.confidence,
                "flags":                "|".join(m.flags),
                "rationale":            m.rationale,
                "model_agreement":      r.model_agreement.get(fw_key, False),
                "reconciliation_note":  m.reconciliation_note or "",
            })
    if not rows:
        return
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved flat CSV → {path}")

def print_summary(results: list[MappingResult]) -> None:
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    for r in results:
        print(f"\n{r.evidence_id}: {r.evidence_text[:80]}")
        for fw_key, m in r.mappings.items():
            agreed = "✓" if r.model_agreement.get(fw_key) else "△"
            print(f"  [{agreed}] {fw_key:30s} conf={m.confidence:.2f}  {m.domains}")
            if m.flags and m.flags != ["not_applicable"]:
                print(f"           flags: {m.flags}")

CSV LOADER

In [19]:
def load_csv(path: str) -> list[EvidenceUnit]:
    units = []
    with open(path, newline="", encoding="utf-8-sig") as f:
        for i, row in enumerate(csv.DictReader(f)):
            units.append(EvidenceUnit(
                id=row.get("id", f"EU_{i+1:04d}"),
                evidence_type=row.get("EvidenceType", "other"),
                relevance_tier=row.get("RelevanceTier", "SUPPORTING"),
                evidence_text=row.get("EvidenceText_Internal", ""),
            ))
    return [u for u in units if u.evidence_text.strip()]

ENTRY POINT

In [20]:
# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    INPUT_FILE  = "merged_rated_0_and_1.csv"
    OUT_JSON    = "mappings_output.json"
    OUT_CSV     = "mappings_output.csv"
    TEST_RUN    = True   # ← set to False to process the full file
    TEST_LIMIT  = 10

    units = load_csv(INPUT_FILE)
    if TEST_RUN:
        units = units[:TEST_LIMIT]
        print(f"TEST RUN — processing first {TEST_LIMIT} of {len(units)} units")
    else:
        print(f"FULL RUN — processing all {len(units)} units")

    print(f"Processing across 5 frameworks...\n")
    results = process_batch(units)
    save_json(results, OUT_JSON)
    save_csv(results,  OUT_CSV)
    print_summary(results)

TEST RUN — processing first 10 of 10 units
Processing across 5 frameworks...


[1/10]

[EU_0001] 'Able to complete writing and submit to employers but self-sa...'
  Stage 1: Decomposing...
  Stage 2: Mapping all frameworks in parallel...
    ✓ stipo
    ✓ broad_domains
    ✓ f13fv
    ✓ bps_4p
    ✓ snomed

────────────────────────────────────────────────────────────
  EU_0001: Able to complete writing and submit to employers but self-sabotaging mode remain
────────────────────────────────────────────────────────────
  [f13fv               ]  conf=0.91  ✓ agree
    domains  : avoidance, goals, self_worth
    rationale: Primary domain is 'avoidance' - the patient demonstrates classic avoidance behavior through self-sabotaging procrastinat
    flags    : ['behavioral_pattern', 'chronic_mechanism', 'occupational_impairment', 'insight_present']
  [broad_domains       ]  conf=0.91  ✓ agree
    domains  : goals_and_priorities, identity_and_self_view, thinking_and_attention
    rationale: PRI